
# Унифицированная валидация симуляции NovaSeq PE150 — PRJEB30386

Ноутбук объединяет два уровня контроля:

1. **Внутренняя корректность симуляции**
   - число R1/R2 и длина reads;
   - сохранение числа молекул при фрагментации;
   - точность распределения IGH / IGK / IGL;
   - выравнивание simulated reads на породившие их фрагменты;
   - выравнивание simulated reads на исходные V–J последовательности;
   - проверка происхождения simulated reads.

2. **Реалистичность относительно PRJEB30386**
   - берётся matched-подвыборка реальных пар из `pr_trimmed`, соответствующих последовательностям,
     прошедшим `post_annotation_filtered`;
   - real и simulated reads выравниваются на один и тот же V–J референс;
   - сравниваются mapping, properly paired, error rate, MAPQ, число отличий от референса,
     soft clipping, длина вставки, GC, профиль качества и повторяемость последовательностей.

## Результаты

Проверяемая ветка задаётся `BCR_SIM_BRANCH` (по умолчанию — текущая ветка NovaSeq). Результаты пишутся в `<ветка>/validation/`: `subset_fastq/`, `fragment_alignment/`, `template_alignment/`. Файл, идентичный уже существующему, не перезаписывается.


In [ ]:

import csv, gzip, hashlib, io, json, math, os, re, shutil, subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysam

DATASET = "PRJEB30386"
BRANCH = os.environ.get("BCR_SIM_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_umimode_in10x_rb3x_ss250")
SAMPLE = "PRJEB30386_all_chains"

SOURCE_RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LOCUS = {
    "ERR3004229": "IGH",
    "ERR3004230": "IGH",
    "ERR3004231": "IGK",
    "ERR3004232": "IGL",
}
RAW_PAIR_COUNTS = {
    "ERR3004229": 1_355_378,
    "ERR3004230": 1_153_931,
    "ERR3004231": 958_261,
    "ERR3004232": 1_085_144,
}

VALIDATION_PAIRS = 250_000
PROFILE_PAIRS = 250_000
NPROC = 8

def resolve_root():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))
    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
        Path.cwd().resolve(),
    ]
    seen = set()
    for start in candidates:
        for root in [start, *start.parents] if start.exists() else [start]:
            if str(root) in seen:
                continue
            seen.add(str(root))
            if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
                return root
    raise FileNotFoundError(
        f"Не найден results/{DATASET}/simulated/{BRANCH}; задайте BCR_VOLUME."
    )

ROOT = resolve_root()
DATASET_DIR = ROOT / "results" / DATASET
SIM_DIR = DATASET_DIR / "simulated" / BRANCH

TRUTH_DIR = SIM_DIR / "00_primary_truth"
PCR1_DIR = SIM_DIR / "01_pcr1"
FRAG_DIR = SIM_DIR / "02_fragmentation"
PCR2_DIR = SIM_DIR / "03_pcr2"
ALLOC_DIR = SIM_DIR / "04_read_allocation"
FASTQ_DIR = SIM_DIR / "06_fastq_pe150"
SIM_QC_DIR = SIM_DIR / "qc"

POST_FILTER_DIR = DATASET_DIR / "post_annotation_filtered"
FILTERED_ASSEMBLED_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"
PR_TRIMMED_DIR = DATASET_DIR / "pr_trimmed" / "fastq"

# СТАРАЯ СХЕМА VALIDATION: не меняем.
VALIDATION_DIR = SIM_DIR / "validation"
SUBSET_DIR = VALIDATION_DIR / "subset_fastq"
FRAG_ALIGN_DIR = VALIDATION_DIR / "fragment_alignment"
TEMPLATE_ALIGN_DIR = VALIDATION_DIR / "template_alignment"
for d in (VALIDATION_DIR, SUBSET_DIR, FRAG_ALIGN_DIR, TEMPLATE_ALIGN_DIR):
    d.mkdir(parents=True, exist_ok=True)

R1 = FASTQ_DIR / f"{SAMPLE}_R1.fastq.gz"
R2 = FASTQ_DIR / f"{SAMPLE}_R2.fastq.gz"
FRAGMENT_FASTA = ALLOC_DIR / f"{SAMPLE}_selected_fragments.fasta"
ALLOCATION_TSV = ALLOC_DIR / f"{SAMPLE}_allocation.tsv"
PRIMARY_TEMPLATES = TRUTH_DIR / f"{SAMPLE}_templates.fasta"
PCR1_TSV = PCR1_DIR / f"{SAMPLE}_pcr1_pool.tsv"
FRAGMENT_TSV = FRAG_DIR / f"{SAMPLE}_fragments.tsv"
FRAGMENT_SUMMARY_TSV = FRAG_DIR / f"{SAMPLE}_fragmentation_summary.tsv"
FINAL_QC = SIM_QC_DIR / "final_qc.tsv"

for p in (
    R1, R2, FRAGMENT_FASTA, ALLOCATION_TSV, PRIMARY_TEMPLATES,
    PCR1_TSV, FRAGMENT_TSV, FRAGMENT_SUMMARY_TSV, FINAL_QC, FILTER_SUMMARY_PATH
):
    if not p.exists():
        raise FileNotFoundError(p)

for run in SOURCE_RUNS:
    for p in (
        FILTERED_ASSEMBLED_DIR / f"{run}_filtered.fastq.gz",
        PR_TRIMMED_DIR / f"{run}_1.pr.fastq.gz",
        PR_TRIMMED_DIR / f"{run}_2.pr.fastq.gz",
    ):
        if not p.exists():
            raise FileNotFoundError(p)

for tool in ("bowtie2", "bowtie2-build", "samtools"):
    if not shutil.which(tool):
        raise RuntimeError(f"{tool} не найден в PATH")

print("ROOT:", ROOT)
print("SIM_DIR:", SIM_DIR)
print("VALIDATION_DIR:", VALIDATION_DIR)


## 1. Унифицированная запись результатов: заменить только если содержимое изменилось

In [ ]:

def sha256_file(path, decompress_gzip=False, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    opener = gzip.open if decompress_gzip else open
    with opener(path, "rb") as src:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def files_identical(a, b):
    a, b = Path(a), Path(b)
    if not a.exists() or not b.exists():
        return False
    if a.suffix == ".gz" and b.suffix == ".gz":
        return sha256_file(a, decompress_gzip=True) == sha256_file(b, decompress_gzip=True)
    return sha256_file(a) == sha256_file(b)

def promote_if_changed(tmp, final, label=None):
    tmp, final = Path(tmp), Path(final)
    label = label or final.name
    if final.exists() and files_identical(tmp, final):
        tmp.unlink()
        print(f"[identical -> keep] {label}")
        return False
    final.parent.mkdir(parents=True, exist_ok=True)
    tmp.replace(final)
    print(f"[updated] {label}")
    return True

def write_text_if_changed(path, text):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(text)
    return promote_if_changed(tmp, path)

def write_json_if_changed(path, obj):
    return write_text_if_changed(
        path,
        json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    )

def write_tsv_if_changed(path, rows, fieldnames=None):
    path = Path(path)
    rows = list(rows)
    if not rows:
        raise ValueError(f"Нет строк для {path}")
    fieldnames = fieldnames or list(rows[0])
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fieldnames, delimiter="\t", lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    return promote_if_changed(tmp, path)

def run_to_log(cmd, log_path):
    log_path = Path(log_path)
    tmp = Path(str(log_path) + ".tmp")
    print("[run]", " ".join(map(str, cmd)), flush=True)
    with open(tmp, "w") as h:
        subprocess.run(list(map(str, cmd)), stdout=h, stderr=subprocess.STDOUT, check=True)
    promote_if_changed(tmp, log_path)

def open_fastq(path, mode="rt"):
    return gzip.open(path, mode)

def read_fastq_record(h):
    a = h.readline()
    if not a:
        return None
    b, c, d = h.readline(), h.readline(), h.readline()
    if not d:
        raise RuntimeError("Обрезанный FASTQ")
    return a, b, c, d

def count_fastq(path):
    n, lengths = 0, set()
    with gzip.open(path, "rt") as h:
        while True:
            rec = read_fastq_record(h)
            if rec is None:
                break
            head, seq, plus, qual = rec
            seq = seq.rstrip("\r\n")
            qual = qual.rstrip("\r\n")
            if not head.startswith("@") or not plus.startswith("+") or len(seq) != len(qual):
                raise RuntimeError(f"Некорректный FASTQ: {path}")
            n += 1
            lengths.add(len(seq))
    return n, sorted(lengths)


## 2. Внутренние инварианты симуляции

In [ ]:

def read_tsv(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

final_rows = read_tsv(FINAL_QC)
if len(final_rows) != 1:
    raise RuntimeError(f"Ожидалась одна строка в {FINAL_QC}")

q = final_rows[0]
expected_pairs = int(q["expected_pairs"])
assert q["sample"] == SAMPLE
assert str(q["valid"]).lower() == "true"

n1, l1 = count_fastq(R1)
n2, l2 = count_fastq(R2)
assert n1 == n2 == expected_pairs
assert l1 == l2 == [150]

pcr1 = read_tsv(PCR1_TSV)
frags = read_tsv(FRAGMENT_TSV)
frag_summary_rows = read_tsv(FRAGMENT_SUMMARY_TSV)
if len(frag_summary_rows) != 1:
    raise RuntimeError(f"Ожидалась одна строка в {FRAGMENT_SUMMARY_TSV}")
frag_summary = frag_summary_rows[0]
alloc = read_tsv(ALLOCATION_TSV)

pcr1_copies = sum(int(r["pcr_copies"]) for r in pcr1 if int(r["pcr_copies"]) > 0)
fragment_input = sum(int(r["fragment_input_copies"]) for r in frags)
assert pcr1_copies == int(frag_summary["pcr1_molecules_total"])
assert fragment_input == int(frag_summary["retained_fragment_molecules"])
assert int(frag_summary["input_nt_mass"]) == int(frag_summary["terminal_nt_mass"])
assert str(frag_summary["fragmentation_nt_conserved"]).lower() in {"true", "1"}

allocated_pairs = sum(int(r["simulated_read_pairs"]) for r in alloc)
assert allocated_pairs == expected_pairs
nonzero_alloc = [r for r in alloc if int(r["simulated_read_pairs"]) > 0]

internal_summary = {
    "expected_pairs": expected_pairs,
    "R1_pairs": n1,
    "R2_pairs": n2,
    "read_length": l1[0],
    "PCR1_molecules": pcr1_copies,
    "fragment_input_molecules": fragment_input,
    "library_input_molecules": int(frag_summary["library_input_molecules"]),
    "retained_fragment_molecules": fragment_input,
    "fragmentation_nt_conserved": True,
    "allocated_pairs": allocated_pairs,
    "selected_fragment_species": len(nonzero_alloc),
}
display(pd.DataFrame(internal_summary.items(), columns=["Показатель", "Значение"]))


## 3. Проверка состава IGH / IGK / IGL

In [ ]:

EXPECTED_LOCUS_PAIRS = {
    "IGH": RAW_PAIR_COUNTS["ERR3004229"] + RAW_PAIR_COUNTS["ERR3004230"],
    "IGK": RAW_PAIR_COUNTS["ERR3004231"],
    "IGL": RAW_PAIR_COUNTS["ERR3004232"],
}
assert sum(EXPECTED_LOCUS_PAIRS.values()) == expected_pairs

observed_locus_pairs = Counter()
for r in alloc:
    observed_locus_pairs[r["locus"]] += int(r["simulated_read_pairs"])

mixture_rows = []
for locus in ("IGH", "IGK", "IGL"):
    exp = EXPECTED_LOCUS_PAIRS[locus]
    obs = observed_locus_pairs[locus]
    row = {
        "locus": locus,
        "expected_pairs": exp,
        "observed_pairs": obs,
        "expected_pct": 100 * exp / expected_pairs,
        "observed_pct": 100 * obs / expected_pairs,
        "delta_pairs": obs - exp,
    }
    mixture_rows.append(row)
    assert obs == exp

write_tsv_if_changed(VALIDATION_DIR / "mixture_qc.tsv", mixture_rows)
display(pd.DataFrame(mixture_rows))


## 4. Детерминированная подвыборка simulated reads

In [ ]:

def write_deterministic_subset(r1, r2, out1, out2, total_pairs, target_pairs):
    if target_pairs is None or target_pairs >= total_pairs:
        return r1, r2, total_pairs

    stride = max(1, total_pairs // target_pairs)
    tmp1 = Path(str(out1) + ".tmp")
    tmp2 = Path(str(out2) + ".tmp")
    selected = 0

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2, \
         gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:
        i = 0
        while True:
            a, b = read_fastq_record(h1), read_fastq_record(h2)
            if a is None or b is None:
                if a is not None or b is not None:
                    raise RuntimeError("R1/R2 заканчиваются в разных местах")
                break
            if i % stride == 0 and selected < target_pairs:
                o1.writelines(a)
                o2.writelines(b)
                selected += 1
            i += 1

    if selected != target_pairs:
        raise RuntimeError(f"Выбрано {selected}, ожидалось {target_pairs}")

    promote_if_changed(tmp1, out1)
    promote_if_changed(tmp2, out2)
    return out1, out2, selected

SUB_R1 = SUBSET_DIR / f"{SAMPLE}_R1.validation.fastq.gz"
SUB_R2 = SUBSET_DIR / f"{SAMPLE}_R2.validation.fastq.gz"

VR1, VR2, validation_pair_count = write_deterministic_subset(
    R1, R2, SUB_R1, SUB_R2, expected_pairs, VALIDATION_PAIRS
)
print("simulated validation pairs:", validation_pair_count)


## 5. Выравнивание simulated reads на точные фрагменты и V–J шаблоны

In [ ]:

def ensure_index(reference, prefix):
    reference = Path(reference)
    prefix = Path(prefix)
    fingerprint = prefix.parent / f"{prefix.name}.reference.sha256"
    current_hash = sha256_file(reference)

    marker = Path(str(prefix) + ".1.bt2")
    marker_l = Path(str(prefix) + ".1.bt2l")
    index_exists = marker.exists() or marker_l.exists()
    fingerprint_ok = fingerprint.exists() and fingerprint.read_text().strip() == current_hash

    if index_exists and fingerprint_ok:
        print("[index unchanged]", prefix)
        return prefix

    for p in prefix.parent.glob(prefix.name + "*.bt2*"):
        p.unlink()
    run_to_log(
        ["bowtie2-build", "--threads", str(NPROC), reference, prefix],
        prefix.parent / "bowtie2_build.log",
    )
    write_text_if_changed(fingerprint, current_hash + "\n")
    return prefix

def align_pair(reference, outdir, label, r1, r2):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    index = ensure_index(reference, outdir / "index")

    tmp_sam = outdir / f".{label}.new.sam"
    tmp_bam = outdir / f".{label}.new.bam"
    tmp_bai = Path(str(tmp_bam) + ".bai")
    final_bam = outdir / f"{label}.bam"
    final_bai = Path(str(final_bam) + ".bai")

    run_to_log([
        "bowtie2", "--very-sensitive-local", "-p", str(NPROC),
        "-x", index, "-1", r1, "-2", r2, "-S", tmp_sam
    ], outdir / f"{label}.bowtie2_align.log")

    subprocess.run(
        ["samtools", "sort", "-@", str(NPROC), "-o", str(tmp_bam), str(tmp_sam)],
        check=True,
    )
    subprocess.run(["samtools", "index", str(tmp_bam)], check=True)
    tmp_sam.unlink(missing_ok=True)

    if final_bam.exists() and files_identical(tmp_bam, final_bam):
        tmp_bam.unlink()
        tmp_bai.unlink(missing_ok=True)
        print(f"[identical -> keep] {final_bam.name}")
    else:
        tmp_bam.replace(final_bam)
        tmp_bai.replace(final_bai)
        print(f"[updated] {final_bam.name}")

    return final_bam

FRAG_BAM = align_pair(
    FRAGMENT_FASTA, FRAG_ALIGN_DIR, "selected_fragments", VR1, VR2
)
TEMPLATE_BAM = align_pair(
    PRIMARY_TEMPLATES, TEMPLATE_ALIGN_DIR, "primary_templates", VR1, VR2
)

print(FRAG_BAM)
print(TEMPLATE_BAM)


In [ ]:

def flagstat_metrics(bam):
    txt = subprocess.run(
        ["samtools", "flagstat", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    mapped_pct = properly_paired_pct = None
    for line in txt.splitlines():
        if " mapped (" in line and "primary mapped" not in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                mapped_pct = float(m.group(1))
        if " properly paired (" in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                properly_paired_pct = float(m.group(1))
    return mapped_pct, properly_paired_pct

def samtools_error_rate(bam):
    txt = subprocess.run(
        ["samtools", "stats", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    for line in txt.splitlines():
        if "error rate:" in line:
            parts = line.split("\t")
            for i, x in enumerate(parts):
                if x.strip() == "error rate:" and i + 1 < len(parts):
                    return float(parts[i + 1])
    return None

frag_mapped, frag_proper = flagstat_metrics(FRAG_BAM)
frag_error = samtools_error_rate(FRAG_BAM)
template_mapped, template_proper = flagstat_metrics(TEMPLATE_BAM)
template_error = samtools_error_rate(TEMPLATE_BAM)

display(pd.DataFrame([
    ["Точные simulated fragments", frag_mapped, frag_proper, frag_error],
    ["Исходные V–J шаблоны", template_mapped, template_proper, template_error],
], columns=["Референс", "Mapped, %", "Properly paired, %", "Error rate"]))


## 6. Соответствие simulated read породившему его фрагменту

In [ ]:

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\r\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

fragment_sequences = dict(iter_fasta(FRAGMENT_FASTA))
def expected_fragment_id(qname):
    # ISS appends _<read_index>_<cpu> to the full fragment identifier.
    # Random-cut fragment IDs themselves end in numeric coordinates, so a
    # legacy _fragN regex is not sufficient; strip exactly the two ISS fields.
    core = re.sub(r"/[12]$", "", qname)
    parts = core.rsplit("_", 2)
    candidate = parts[0] if len(parts) == 3 else None
    return candidate if candidate in fragment_sequences else None

def fragment_origin_metrics(bam, max_primary_reads=500_000):
    parsed = exact = equivalent = primary = 0
    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue
            primary += 1
            exp = expected_fragment_id(r.query_name)
            if exp is not None and exp in fragment_sequences:
                parsed += 1
                if exp == r.reference_name:
                    exact += 1
                    equivalent += 1
                elif (
                    r.reference_name in fragment_sequences
                    and fragment_sequences[exp] == fragment_sequences[r.reference_name]
                ):
                    equivalent += 1
            if primary >= max_primary_reads:
                break
    return {
        "primary_reads_checked": primary,
        "qname_origin_parse_rate": parsed / primary if primary else None,
        "exact_fragment_origin_rate": exact / parsed if parsed else None,
        "sequence_equivalent_fragment_origin_rate": equivalent / parsed if parsed else None,
    }

fragment_origin = fragment_origin_metrics(FRAG_BAM)
fragment_origin



## 7. Matched-подвыборка реальных paired-end reads

`post_annotation_filtered` содержит уже собранные pRESTO-последовательности, поэтому их нельзя
непосредственно сравнивать с simulated PE150 по paired-end alignment.

Используем идентификаторы собранных последовательностей, прошедших post-annotation filter,
и возвращаемся к соответствующим парам в `pr_trimmed/fastq`.


In [ ]:

FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())

def normalized_coord(header):
    x = header.strip()
    if x.startswith("@"):
        x = x[1:]
    x = x.split(None, 1)[0]
    x = x.split("|", 1)[0]
    x = re.sub(r"/[12]$", "", x)
    return x

def proportional_quotas(total):
    denom = sum(RAW_PAIR_COUNTS.values())
    exact = {r: total * RAW_PAIR_COUNTS[r] / denom for r in SOURCE_RUNS}
    q = {r: int(math.floor(v)) for r, v in exact.items()}
    remainder = total - sum(q.values())
    for r in sorted(SOURCE_RUNS, key=lambda x: exact[x] - q[x], reverse=True)[:remainder]:
        q[r] += 1
    return q

REAL_QUOTAS = proportional_quotas(validation_pair_count)

def select_filtered_coordinates(run, target):
    fq = FILTERED_ASSEMBLED_DIR / f"{run}_filtered.fastq.gz"
    total = int(FILTER_SUMMARY["samples"][run]["passed"])
    stride = max(1, total // target)
    selected = []
    with gzip.open(fq, "rt") as h:
        i = 0
        while True:
            rec = read_fastq_record(h)
            if rec is None:
                break
            if i % stride == 0 and len(selected) < target:
                selected.append(normalized_coord(rec[0]))
            i += 1
    if len(selected) != target:
        raise RuntimeError(f"{run}: выбрано {len(selected)} вместо {target}")
    return set(selected)

selected_coordinates = {
    run: select_filtered_coordinates(run, REAL_QUOTAS[run])
    for run in SOURCE_RUNS
}
print({run: len(ids) for run, ids in selected_coordinates.items()})


In [ ]:

REAL_R1 = SUBSET_DIR / "PRJEB30386_real_postfilter_R1.validation.fastq.gz"
REAL_R2 = SUBSET_DIR / "PRJEB30386_real_postfilter_R2.validation.fastq.gz"

def build_real_matched_subset():
    tmp1 = Path(str(REAL_R1) + ".tmp")
    tmp2 = Path(str(REAL_R2) + ".tmp")
    found = Counter()

    with gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:
        for run in SOURCE_RUNS:
            wanted = selected_coordinates[run]
            r1 = PR_TRIMMED_DIR / f"{run}_1.pr.fastq.gz"
            r2 = PR_TRIMMED_DIR / f"{run}_2.pr.fastq.gz"
            with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2:
                while True:
                    a, b = read_fastq_record(h1), read_fastq_record(h2)
                    if a is None or b is None:
                        if a is not None or b is not None:
                            raise RuntimeError(f"{run}: R1/R2 имеют разную длину")
                        break
                    ca, cb = normalized_coord(a[0]), normalized_coord(b[0])
                    if ca != cb:
                        raise RuntimeError(f"{run}: несинхронная пара {ca} != {cb}")
                    if ca in wanted:
                        o1.writelines(a)
                        o2.writelines(b)
                        found[run] += 1

    for run in SOURCE_RUNS:
        if found[run] != REAL_QUOTAS[run]:
            raise RuntimeError(
                f"{run}: найдено {found[run]} matched-пар, ожидалось {REAL_QUOTAS[run]}. "
                "Проверьте сохранение SRA coordinate между pr_trimmed и assembled."
            )

    promote_if_changed(tmp1, REAL_R1)
    promote_if_changed(tmp2, REAL_R2)
    return dict(found)

real_subset_counts = build_real_matched_subset()
print(real_subset_counts)


## 8. Real и simulated на одном V–J референсе

In [ ]:

# Simulated V–J alignment уже рассчитан выше и остаётся в старом файле:
# validation/template_alignment/primary_templates.bam

REAL_TEMPLATE_BAM = align_pair(
    PRIMARY_TEMPLATES,
    TEMPLATE_ALIGN_DIR,
    "real_postfilter_primary_templates",
    REAL_R1,
    REAL_R2,
)

def softclip_bases(read):
    if not read.cigartuples:
        return 0
    return sum(length for op, length in read.cigartuples if op == 4)

def bam_read_metrics(bam, max_primary_reads=500_000):
    rows = []
    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_secondary or r.is_supplementary or r.is_unmapped:
                continue
            qlen = r.query_length or 0
            nm = r.get_tag("NM") if r.has_tag("NM") else np.nan
            locus = None
            if r.reference_name:
                m = re.search(r"_(IGH|IGK|IGL)_tpl_", r.reference_name)
                locus = m.group(1) if m else None
            rows.append({
                "mapq": r.mapping_quality,
                "nm_per_base": (nm / qlen) if qlen and not pd.isna(nm) else np.nan,
                "softclip_fraction": softclip_bases(r) / qlen if qlen else np.nan,
                "insert_size": (
                    abs(r.template_length)
                    if r.is_read1 and r.is_proper_pair and r.template_length
                    else np.nan
                ),
                "locus": locus,
            })
            if len(rows) >= max_primary_reads:
                break
    return pd.DataFrame(rows)

real_reads = bam_read_metrics(REAL_TEMPLATE_BAM)
sim_reads = bam_read_metrics(TEMPLATE_BAM)

def alignment_summary_row(label, bam, df):
    mapped, proper = flagstat_metrics(bam)
    return {
        "dataset": label,
        "mapped_pct": mapped,
        "properly_paired_pct": proper,
        "error_rate": samtools_error_rate(bam),
        "median_mapq": float(df["mapq"].median()),
        "mean_nm_per_base": float(df["nm_per_base"].mean()),
        "mean_softclip_fraction": float(df["softclip_fraction"].mean()),
        "median_insert_size": float(df["insert_size"].dropna().median()),
        "primary_mapped_reads_profiled": int(len(df)),
    }

alignment_rows = [
    alignment_summary_row("real_postfilter", REAL_TEMPLATE_BAM, real_reads),
    alignment_summary_row("simulated", TEMPLATE_BAM, sim_reads),
]
write_tsv_if_changed(VALIDATION_DIR / "real_vs_simulated_alignment.tsv", alignment_rows)
display(pd.DataFrame(alignment_rows))


## 9. GC, качество оснований и повторяемость последовательностей

In [ ]:

def pair_profiles(r1, r2, max_pairs=PROFILE_PAIRS, max_cycle=150):
    qsum1 = np.zeros(max_cycle, dtype=float)
    qsum2 = np.zeros(max_cycle, dtype=float)
    qn1 = np.zeros(max_cycle, dtype=int)
    qn2 = np.zeros(max_cycle, dtype=int)
    gc = []
    pair_hashes = Counter()
    pairs = 0

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2:
        while pairs < max_pairs:
            a, b = read_fastq_record(h1), read_fastq_record(h2)
            if a is None or b is None:
                break
            s1, q1 = a[1].strip().upper(), a[3].strip()
            s2, q2 = b[1].strip().upper(), b[3].strip()

            for arr, cnt, q in ((qsum1, qn1, q1), (qsum2, qn2, q2)):
                for i, ch in enumerate(q[:max_cycle]):
                    arr[i] += ord(ch) - 33
                    cnt[i] += 1

            seq = s1 + s2
            if seq:
                gc.append(100 * (seq.count("G") + seq.count("C")) / len(seq))
            pair_hashes[hashlib.sha1((s1 + "|" + s2).encode()).hexdigest()] += 1
            pairs += 1

    return {
        "pairs": pairs,
        "q1": np.divide(qsum1, qn1, out=np.full(max_cycle, np.nan), where=qn1 > 0),
        "q2": np.divide(qsum2, qn2, out=np.full(max_cycle, np.nan), where=qn2 > 0),
        "gc": np.asarray(gc),
        "duplicate_pair_fraction": (
            sum(v for v in pair_hashes.values() if v > 1) / pairs if pairs else np.nan
        ),
    }

real_prof = pair_profiles(REAL_R1, REAL_R2)
sim_prof = pair_profiles(VR1, VR2)

profile_rows = [
    {
        "dataset": "real_postfilter",
        "pairs": real_prof["pairs"],
        "mean_gc_pct": float(np.mean(real_prof["gc"])),
        "median_gc_pct": float(np.median(real_prof["gc"])),
        "duplicate_pair_fraction": float(real_prof["duplicate_pair_fraction"]),
        "mean_q_R1": float(np.nanmean(real_prof["q1"])),
        "mean_q_R2": float(np.nanmean(real_prof["q2"])),
    },
    {
        "dataset": "simulated",
        "pairs": sim_prof["pairs"],
        "mean_gc_pct": float(np.mean(sim_prof["gc"])),
        "median_gc_pct": float(np.median(sim_prof["gc"])),
        "duplicate_pair_fraction": float(sim_prof["duplicate_pair_fraction"]),
        "mean_q_R1": float(np.nanmean(sim_prof["q1"])),
        "mean_q_R2": float(np.nanmean(sim_prof["q2"])),
    },
]
write_tsv_if_changed(VALIDATION_DIR / "real_vs_simulated_sequence_qc.tsv", profile_rows)
display(pd.DataFrame(profile_rows))


In [ ]:

def save_figure_if_changed(fig, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp.png")
    fig.savefig(tmp, dpi=160, bbox_inches="tight")
    promote_if_changed(tmp, path)

cycles = np.arange(1, 151)

fig = plt.figure(figsize=(10, 5))
plt.plot(cycles, real_prof["q1"], label="Real R1")
plt.plot(cycles, sim_prof["q1"], label="Simulated R1")
plt.xlabel("Позиция в read")
plt.ylabel("Средний Phred")
plt.title("R1: качество оснований")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "quality_profile_R1.png")
plt.show()

fig = plt.figure(figsize=(10, 5))
plt.plot(cycles, real_prof["q2"], label="Real R2")
plt.plot(cycles, sim_prof["q2"], label="Simulated R2")
plt.xlabel("Позиция в read")
plt.ylabel("Средний Phred")
plt.title("R2: качество оснований")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "quality_profile_R2.png")
plt.show()

fig = plt.figure(figsize=(9, 5))
plt.hist(real_prof["gc"], bins=60, density=True, alpha=0.5, label="Real")
plt.hist(sim_prof["gc"], bins=60, density=True, alpha=0.5, label="Simulated")
plt.xlabel("GC, %")
plt.ylabel("Плотность")
plt.title("GC-состав")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "gc_distribution.png")
plt.show()


## 10. Состав локусов по результатам общего V–J выравнивания

In [ ]:

def locus_rows(df, label):
    counts = df["locus"].dropna().value_counts()
    total = counts.sum()
    return [
        {
            "dataset": label,
            "locus": locus,
            "mapped_reads": int(n),
            "pct": float(100 * n / total) if total else np.nan,
        }
        for locus, n in counts.items()
    ]

locus_rows_all = (
    locus_rows(real_reads, "real_postfilter")
    + locus_rows(sim_reads, "simulated")
)
write_tsv_if_changed(
    VALIDATION_DIR / "real_vs_simulated_locus_alignment.tsv",
    locus_rows_all,
)
display(pd.DataFrame(locus_rows_all))



## 11. Единая итоговая сводка

`validation_summary.json` остаётся главным итоговым файлом старой схемы.
Старые поля сохраняются; к ним добавляется блок `realism_comparison`.


In [ ]:

template_metrics = {
    "mapped_pct": template_mapped,
    "properly_paired_pct": template_proper,
    "error_rate": template_error,
}

summary = {
    **internal_summary,
    "branch": BRANCH,
    "sample": SAMPLE,
    "validation_pairs": validation_pair_count,
    "locus_pairs": dict(observed_locus_pairs),
    "fragment_alignment": {
        "mapped_pct": frag_mapped,
        "properly_paired_pct": frag_proper,
        "error_rate": frag_error,
        **fragment_origin,
    },
    "template_alignment": template_metrics,
    "realism_comparison": {
        "real_subset_pairs": int(sum(real_subset_counts.values())),
        "real_subset_pairs_by_run": real_subset_counts,
        "alignment": {
            row["dataset"]: {k: v for k, v in row.items() if k != "dataset"}
            for row in alignment_rows
        },
        "sequence_qc": {
            row["dataset"]: {k: v for k, v in row.items() if k != "dataset"}
            for row in profile_rows
        },
        "interpretation": (
            "Внутреннее выравнивание simulated->truth проверяет корректность симуляции; "
            "real-vs-simulated на одном V-J референсе проверяет сходство поведения reads. "
            "Полная оценка пригодности для benchmark BCR reconstruction дополнительно "
            "требует запуска TRUST4 и сравнения реконструкции с известным truth."
        ),
    },
}

write_json_if_changed(VALIDATION_DIR / "validation_summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))



## Интерпретация

Сильная валидация требует одновременно двух условий:

1. **Simulated → truth**
   - высокая доля mapping;
   - высокая доля properly paired;
   - низкая частота ошибок;
   - отсутствие нарушения состава IGH/IGK/IGL;
   - сохранение внутренних инвариантов симуляции.

2. **Real ↔ simulated**
   - сопоставимые alignment-метрики на одном V–J референсе;
   - сопоставимые MAPQ, ошибки и soft clipping;
   - сопоставимый GC-профиль и качество оснований;
   - отсутствие радикального расхождения по повторяемости reads.

100% mapping simulated reads сам по себе не означает полной реалистичности:
симулированные reads построены из известного отфильтрованного truth и по определению могут
быть проще реальных reads. Поэтому real-vs-simulated сравнение включено в этот же ноутбук.
